In [1]:
%%writefile preprocessing.h

#pragma once
#ifndef PREPROCESS_H
#define PREPROCESS_H

void preprocess(char* RefSeq, char* ReadSeq, int ReadLength)
{
    int i, index = 0;

#define ENCODE_BASE(b) \
        ((b) == 'A' || (b) == 'a' ? 0b0001 : \
         (b) == 'C' || (b) == 'c' ? 0b0010 : \
         (b) == 'G' || (b) == 'g' ? 0b0011 : \
         (b) == 'T' || (b) == 't' ? 0b0100 : \
         (b) == 'N' || (b) == 'n' ? 0b0101 : 0x00)

    for (i = 0; i < ReadLength; i += 2) {
        unsigned char base1_r = ENCODE_BASE(ReadSeq[i]);
        unsigned char base1_f = ENCODE_BASE(RefSeq[i]);

        unsigned char base2_r = 0;
        unsigned char base2_f = 0;

        if (i + 1 < ReadLength) {
            base2_r = ENCODE_BASE(ReadSeq[i + 1]);
            base2_f = ENCODE_BASE(RefSeq[i + 1]);
        }
        else {
            base2_r = 0x0F; // Padding for odd length
            base2_f = 0x0F; // Padding for odd length
        }

        // Pack two 4-bit bases into one byte
        ReadSeq[index] = (char)((base1_r << 4) | base2_r);
        RefSeq[index] = (char)((base1_f << 4) | base2_f);
        index++;
    }

#undef ENCODE_BASE
}

#endif // PREPROCESS_H

Overwriting preprocessing.h


In [2]:
%%writefile checkpointfull.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <time.h>
#include "preprocessing.h"

// ==============================================================================
// TIMING INFRASTRUCTURE (Must match ASM layout)
// ==============================================================================

// 24 bytes per timer
typedef struct {
    uint64_t start_cycles;  // 8 bytes
    uint64_t total_cycles;  // 8 bytes
    uint64_t call_count;    // 8 bytes (using uint64_t to match 64-bit ASM long)
} Timer;

// Layout matching ASM offsets:
// 0:  preprocess
// 24: main_diagonal
// 48: right_diagonal
// 72: left_diagonal
// 96: snake_total
typedef struct {
    Timer preprocess_timer;       // Offset 0
    Timer main_diagonal_timer;    // Offset 24
    Timer right_diagonal_timer;   // Offset 48
    Timer left_diagonal_timer;    // Offset 72
    Timer snake_total_timer;      // Offset 96
} TimingData;

// Global instance shared with ASM
TimingData timing_data = {0};

// Helper to convert CPU cycles to seconds (Assuming 4GHz CPU, adjust if needed)
static double cycles_to_seconds(uint64_t cycles) {
    static double cpu_freq_ghz = 4.0; 
    return cycles / (cpu_freq_ghz * 1e9);
}

void print_timing_breakdown() {
    printf("\n=== ASM INTERNAL TIMING BREAKDOWN ===\n");
    printf("%-25s %12s %12s %15s\n", "Component", "Total (s)", "Calls", "Avg Cycles");
    printf("----------------------------------------------------------------------------\n");

    #define PRINT_TIMER(name, t) \
        printf("%-25s %12.6f %12lu %15.0f\n", \
               name, \
               cycles_to_seconds(t.total_cycles), \
               t.call_count, \
               t.call_count > 0 ? (double)t.total_cycles / t.call_count : 0.0)

    PRINT_TIMER("Total SneakySnake", timing_data.snake_total_timer);
    PRINT_TIMER("  Main Diagonal",   timing_data.main_diagonal_timer);
    PRINT_TIMER("  Right (Upper) Diag", timing_data.right_diagonal_timer);
    PRINT_TIMER("  Left (Lower) Diag",  timing_data.left_diagonal_timer);
    
    // Calculate overhead (Total - components)
    uint64_t components = timing_data.main_diagonal_timer.total_cycles +
                          timing_data.right_diagonal_timer.total_cycles +
                          timing_data.left_diagonal_timer.total_cycles;
    
    if (timing_data.snake_total_timer.total_cycles > components) {
        uint64_t overhead = timing_data.snake_total_timer.total_cycles - components;
        printf("%-25s %12.6f %12s %15s\n", 
               "  Other/Overhead", cycles_to_seconds(overhead), "-", "-");
    }
    printf("----------------------------------------------------------------------------\n");
}

void reset_asm_timers() {
    memset(&timing_data, 0, sizeof(TimingData));
}


// Helper function to extract nibble from packed byte array
static inline uint8_t get_nibble(uint8_t* seq, int base_index) {
    int byte_idx = base_index / 2;
    int is_lower = base_index % 2;
    if (is_lower) {
        return seq[byte_idx] & 0x0F;  // Lower nibble
    } else {
        return (seq[byte_idx] >> 4) & 0x0F;  // Upper nibble
    }
}

// C implementation for comparison - works with nibble-packed data
int SneakySnake_C(int ReadLength, uint8_t* RefSeq, uint8_t* ReadSeq, 
                  int EditThreshold, int IterationNo)
{
    // ReadLength is the ORIGINAL length (number of bases), not packed length
    int index = 0;
    int Edits = 0;
    int roundsNo = 1;
    
    while (index < ReadLength) {
        if (roundsNo > IterationNo) {
            return 0;
        }
        
        if (Edits > EditThreshold) {
            return 0;
        }
        
        int GlobalCount = 0;
        
        // Main diagonal - compare nibbles
        int count = 0;
        for (int n = index; n < ReadLength; n++) {
            uint8_t read_base = get_nibble(ReadSeq, n);
            uint8_t ref_base = get_nibble(RefSeq, n);
            if (read_base != ref_base) {
                break;
            }
            count++;
        }
        GlobalCount = count;
        
        if (GlobalCount == (ReadLength - index)) {
            return (Edits <= EditThreshold) ? 1 : 0;
        }
        
        // Upper and lower diagonals
        for (int e = 1; e <= EditThreshold; e++) {
            count = 0;
            
            // Upper diagonal (shift read right)
            for (int n = index; n < ReadLength; n++) {
                if (n < e) break;
                uint8_t read_base = get_nibble(ReadSeq, n - e);
                uint8_t ref_base = get_nibble(RefSeq, n);
                if (read_base != ref_base) break;
                count++;
            }
            
            if (count > GlobalCount) {
                GlobalCount = count;
            }
            if (count == (ReadLength - index)) {
                return (Edits <= EditThreshold) ? 1 : 0;
            }
            
            count = 0;
            
            // Lower diagonal (shift read left)
            for (int n = index; n < ReadLength; n++) {
                if (n > ReadLength - e - 1) break;
                uint8_t read_base = get_nibble(ReadSeq, n + e);
                uint8_t ref_base = get_nibble(RefSeq, n);
                if (read_base != ref_base) break;
                count++;
            }
            
            if (count > GlobalCount) {
                GlobalCount = count;
            }
            if (count == (ReadLength - index)) {
                return (Edits <= EditThreshold) ? 1 : 0;
            }
        }
        
        index += GlobalCount;
        
        if (index < ReadLength) {
            Edits++;
            index++;
        }
        
        roundsNo++;
    }
    
    return (Edits <= EditThreshold) ? 1 : 0;
}

// External assembly function
extern uint64_t SneakySnake(uint64_t ReadLength, uint8_t* RefSeq, 
                             uint8_t* ReadSeq, uint64_t EditThreshold,
                             uint64_t IterationNo);

extern uint64_t current_position;
extern uint64_t current_edits;
extern uint64_t mismatch_count;
extern uint64_t safety_counter;

// calculate edit distance (levenshtein distance)
int calculate_edit_distance(const char* s1, const char* s2, int len) {
    int m = len, n = len;
    int dp[m + 1][n + 1];
    
    for (int i = 0; i <= m; i++) dp[i][0] = i;
    for (int j = 0; j <= n; j++) dp[0][j] = j;
    
    for (int i = 1; i <= m; i++) {
        for (int j = 1; j <= n; j++) {
            if (s1[i-1] == s2[j-1]) {
                dp[i][j] = dp[i-1][j-1];
            } else {
                dp[i][j] = 1 + (dp[i-1][j] < dp[i][j-1] ? 
                               (dp[i-1][j] < dp[i-1][j-1] ? dp[i-1][j] : dp[i-1][j-1]) :
                               (dp[i][j-1] < dp[i-1][j-1] ? dp[i][j-1] : dp[i-1][j-1]));
            }
        }
    }
    return dp[m][n];
}

int main(int argc, char *argv[]) {
    if (argc < 4) {
        printf("Usage: %s <file> <threshold> <seq_len> [limit] [debug_limit]\n", argv[0]);
        return 1;
    }
    
    const char* filename = argv[1];
    int threshold = atoi(argv[2]);
    int fixed_len = atoi(argv[3]); 
    int limit = (argc >= 5) ? atoi(argv[4]) : 30000;
    int debug_limit = (argc >= 6) ? atoi(argv[5]) : 5;
    
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open file %s\n", filename);
        return 1;
    }
    
    printf("Loading sequences (Fixed Len=%d, Limit=%d)...\n", fixed_len, limit);
    
    char **read_orig = malloc(limit * sizeof(char*));
    char **ref_orig = malloc(limit * sizeof(char*));
    uint8_t **read_enc = malloc(limit * sizeof(uint8_t*));
    uint8_t **ref_enc = malloc(limit * sizeof(uint8_t*));
    int *lengths = malloc(limit * sizeof(int));
    int *edit_dists = malloc(limit * sizeof(int));
    
    char *line = NULL;
    size_t cap = 0;
    int count = 0;
    int ground_truth_matches = 0;
    int skipped = 0;

     while (getline(&line, &cap, file) != -1 && count < limit) {
        char *read_seq = line;
        char *ref_seq = strpbrk(line, "\t ");
        if (!ref_seq) continue;
        *ref_seq = '\0';
        ref_seq++;
        while (*ref_seq == ' ' || *ref_seq == '\t') ref_seq++;
        ref_seq[strcspn(ref_seq, "\r\n")] = 0;
        
        int actual_len = strlen(read_seq);
        if (actual_len < fixed_len) {
            skipped++;
            continue; 
        }
        
        int len = fixed_len;
        read_seq[len] = '\0';
        ref_seq[len] = '\0';
        
        lengths[count] = len;
        read_orig[count] = strdup(read_seq);
        ref_orig[count] = strdup(ref_seq);
        
        read_enc[count] = calloc(len + 64, 1);
        ref_enc[count]  = calloc(len + 64, 1);

        memcpy(read_enc[count], read_seq, len);
        memcpy(ref_enc[count], ref_seq, len);

        preprocess((char*)ref_enc[count], (char*)read_enc[count], len);

        //edit_dists[count] = calculate_edit_distance(read_seq, ref_seq, len);
        //if (edit_dists[count] <= threshold && len <= 2000) {
          //  ground_truth_matches++;
        //}

    if (len <= 2000) {
        edit_dists[count] = calculate_edit_distance(read_seq, ref_seq, len);
        if (edit_dists[count] <= threshold) ground_truth_matches++;
    } else {
        edit_dists[count] = -1; // skip huge O(n^2) DP
    }


        count++;
        if (count % 500 == 0) {
            printf("Loaded %d sequences...\r", count);
            fflush(stdout);
        }
    }
    
    fclose(file);
    free(line);
    
    // Debug first few (Timers disabled here to avoid polluting benchmark stats)
    printf("\n=== DEBUGGING FIRST %d SEQUENCES ===\n", debug_limit);
    
    for (int i = 0; i < debug_limit && i < count; i++) {
        if (lengths[i] > 5000){
            printf("\n[%d] Len=%d (skipping C debug for long length)\n", i, lengths[i]);
            continue;
        }
        int c_result = SneakySnake_C(lengths[i], ref_enc[i], read_enc[i], 
                                     threshold, lengths[i] * 2);
        
        // Reset counters/timers just for debug clarity
        current_position = 0;
        current_edits = 0;
        mismatch_count = 0;
        safety_counter = 0;
        
        int asm_result = SneakySnake(lengths[i], ref_enc[i], read_enc[i], 
                                     threshold, lengths[i] * 2);
        
        printf("\n[%d] Len=%d\n", i, lengths[i]);
        printf("    C:    %s\n", c_result ? "ACCEPT" : "REJECT");
        printf("    ASM:  %s\n", asm_result ? "ACCEPT" : "REJECT");
        
        if (c_result != asm_result) {
            printf("    *** C AND ASM DISAGREE! ***\n");
        }
    }
    
    // Clear timers before full benchmark
    reset_asm_timers();
    
    // Full benchmark
    printf("\n=== RUNNING FULL BENCHMARK ===\n");
    int c_matches = 0;
    int asm_matches = 0;
    
    // C implementation
    clock_t c_start = clock();
    for (int i = 0; i < count; i++) {
        int result = SneakySnake_C(lengths[i], ref_enc[i], read_enc[i], 
                                   threshold, lengths[i] * 2);
        if (result) c_matches++;
    }
    clock_t c_end = clock();
    double c_time = ((double)(c_end - c_start)) / CLOCKS_PER_SEC;
    
    // ASM implementation (With Timing Instrumentation)
    clock_t asm_start = clock();
    for (int i = 0; i < count; i++) {
        current_position = 0;
        current_edits = 0;
        mismatch_count = 0;
        safety_counter = 0;
        
        int result = SneakySnake(lengths[i], ref_enc[i], read_enc[i], 
                                 threshold, lengths[i] * 2);
        if (result) asm_matches++;
    }
    clock_t asm_end = clock();
    double asm_time = ((double)(asm_end - asm_start)) / CLOCKS_PER_SEC;
    
    printf("\n==================================================\n");
    printf("                  BENCHMARK SUMMARY               \n");
    printf("==================================================\n");
    printf("Total Pairs              : %d\n", count);
    printf("Fixed Length             : %d\n", fixed_len);
    printf("Threshold                : %d\n", threshold);
    printf("\n");
    printf("C Implementation:\n");
    printf("  Accepted               : %d\n", c_matches);
    printf("  Time                   : %.4f seconds\n", c_time);
    printf("\n");
    printf("ASM Implementation:\n");
    printf("  Accepted               : %d\n", asm_matches);
    printf("  Time                   : %.4f seconds\n", asm_time);
    if (asm_time > 0)
        printf("  Speedup vs C           : %.2fx\n", c_time / asm_time);
    
    // NEW: Print detailed breakdown
    print_timing_breakdown();
    
    if (c_matches != asm_matches) {
        printf("\nWARNING: C and ASM implementations disagree!\n");
        printf("C accepted %d, ASM accepted %d (diff = %d)\n", 
               c_matches, asm_matches, abs(c_matches - asm_matches));
    }
    
    // Cleanup
    for (int i = 0; i < count; i++) {
        free(read_orig[i]);
        free(ref_orig[i]);
        free(read_enc[i]);
        free(ref_enc[i]);
    }
    free(read_orig);
    free(ref_orig);
    free(read_enc);
    free(ref_enc);
    free(lengths);
    free(edit_dists);
    
    return 0;
}

Overwriting checkpointfull.c


In [3]:
%%writefile checkpointavxfull.asm

default rel
bits 64

section .data
global SneakySnake
global current_position
global current_edits
global mismatch_count
global safety_counter

current_position dq 0
current_edits dq 0
mismatch_count dq 0
safety_counter dq 0

section .text
global SneakySnake

; -------------------------------------------------------------------------------------
; sneaky snake with avx-512 parallel diagonal checking
; avx-512 allows us to compare 64 bytes (128 nibbles) in a single instruction
; instead of checking each nibble one at a time in a loop

; in original code, what happens is
; for each position in sequence:
    ; extract nibble from read
    ; extract nibble from ref
    ; compare
    ; if mismatch: stop
    ; else: continue

; OKAY PLS READ THIS so what happened and the difference with the old code is that
    ; rdi - editThreshold 
    ; rsi - ReadSeq -> r11
    ; rdx - RefSeq -> r12
    ; rcx - ReadLength -> r13
    ; r8 - Iteration -> Iteration is now stored in stack
    ; r9 - shift amount for diagonals
    ; r15 - buffer offset for diagonal storage
    ; rax - match counter in diagonal functions
    ; rbx - current position calculations

; DATA STORAGE
; in the old code we used 4 sets of 64-byte buffers right_diag_read/ref, left_diag_read/ref
; this time the results are stored only in registers and stakc variables

; NIBBLE EXTRACTION
; same approach pa rin naman for byte-aligned data but it uses scalar for unaligned

; MATCH COUNTING
; we used tzcnt to count leading matches, now we check if bytes matched first, if all
; matched then add 128 and continue, if partial, count bit by bit

; DIAGONAL SHIFTING LOGIC
; before we used first-block handling, now this one if we can't use avx-512 it falls back to scalar
; doesn't use special cases anymore. it just checks if shifted positions are valid

; MAIN LOOP PROGRESSION
; this one goes straight to finding the longest match, then mismatch it checks diagonals na
; HAY PUCHHH

; GLOBAL VARIABLES
; doesn't have global counter anymore because it uses local variable r14 to track best match

; OK WAIT CZAR READ THIS ACTUALLY  
; this one prioritizes correctness more than speed kaya mas maraming SCALAR dito and less special
; cases for nibbles oki HUHUHU para it's accurate and sir rog happy yehey
; -------------------------------------------------------------------------------------

SneakySnake:
    push    rbp
    mov     rbp, rsp
    push    rbx
    push    r12
    push    r13
    push    r14
    push    r15
    sub     rsp, 128

    ; parameters:
    ; rdi = readlength (in nibbles)
    ; rsi = refseq
    ; rdx = readseq
    ; rcx = editthreshold
    ; r8  = iterationno
    
    mov     r13, rdi              ; readlength
    mov     r12, rsi              ; refseq pointer
    mov     r11, rdx              ; readseq pointer
    mov     r10, rcx              ; editthreshold
    mov     [rbp-8], r8           ; iterationno

    xor     r15, r15              ; index = 0 (our current position in the sequences)
    
.main_loop:
    ; check if we've processed the entire read
    cmp     r15, r13 ; r15 = position if it's equal to readlength
    jae     .accept
    
    ; safety check to prevent infinite loops
    inc     qword [safety_counter]
    mov     rax, [safety_counter]
    cmp     rax, [rbp-8]
    jg      .reject
    
    ; check if we've exceeded the edit threshold (too many mismatches)
    mov     rax, [current_edits]
    cmp     rax, r10
    jg      .reject
    
    ; calculate how many nibbles remain to check
    mov     r14, r13
    sub     r14, r15              ; remaining = readlength - index
    
    ; check main diagonal with avx-512
    ; we compare many nibbles at once
    call    .check_main_diagonal_avx512
    mov     r14, rax              ; globalcount = main diagonal matches
    
    ; if main diagonal matches everything remaining, we're done!
    mov     rbx, r13
    sub     rbx, r15
    cmp     rax, rbx
    jae     .matched_all_remaining
    
    ; store best diagonal info
    xor     r9, r9                ; best_shift = 0 (main diagonal)
    mov     [rbp-24], r9
    
    ; now check all shifted diagonals (for insertions and deletions)
    mov     r8, 1                 ; shift = 1


; shift_loop basically handles all diagonals. it checks the upper (shift right = deletion) and lower (shift left = insertion)
; example: if editThreshold = 5, then algorithm checks
    ; main diagonall = shift 0
    ; upper diagonals = shifts (1, 2, 3, 4, 5) <- 5 deletions
    ; lower diagonals = shifts (-1, -2, -3, -4, -5) <- 5 insertions TAMA BA TO HELPPPPP
.shift_loop:
    cmp     r8, r10
    ja      .shift_done
    
    ; check upper diagonal (deletion in reference)
    ; this means the read is missing a base that's in the reference
    mov     r9, r8
    call    .check_upper_diagonal_avx512
    
    ; if this matches everything, take it immediately
    mov     rbx, r13
    sub     rbx, r15
    cmp     rax, rbx
    jae     .matched_all_remaining
    
    ; update if this diagonal is better
    cmp     rax, r14
    jbe     .check_lower
    
    mov     r14, rax
    mov     r9, r8
    mov     [rbp-24], r9
    
.check_lower:
    ; check lower diagonal (insertion in reference)
    ; this means the read has an extra base that's not in the reference
    mov     r9, r8
    call    .check_lower_diagonal_avx512
    
    ; if this matches everything, take it immediately
    mov     rbx, r13
    sub     rbx, r15
    cmp     rax, rbx
    jae     .matched_all_remaining
    
    ; update if this diagonal is better
    cmp     rax, r14
    jbe     .next_shift
    
    mov     r14, rax
    mov     r9, r8
    neg     r9
    mov     [rbp-24], r9
    
.next_shift:
    inc     r8
    jmp     .shift_loop
    
.shift_done:
    ; advance by globalcount (skip all the matched positions)
    add     r15, r14
    
    ; if we reached the end, loop back to check completion
    cmp     r15, r13
    jae     .main_loop
    
    ; not at end: we hit a mismatch/error
    ; add 1 edit and skip the error position
    inc     qword [current_edits]
    inc     r15
    
    jmp     .main_loop

.matched_all_remaining:
    mov     r15, r13
    jmp     .main_loop

.accept:
    mov     [current_position], r13
    mov     rax, [current_edits]
    cmp     rax, r10
    jg      .reject
    mov     rax, 1                ; return 1 (success)
    jmp     .end

.reject:
    xor     rax, rax              ; return 0 (failure)
    jmp     .end

.no_avx512:
    xor     rax, rax              ; return 0 if no avx-512 support

.end:
    add     rsp, 128
    pop     r15
    pop     r14
    pop     r13
    pop     r12
    pop     rbx
    leave
    ret

; -------------------------------------------------------------------------------------
; avx-512 optimized main diagonal checker
; this is the core parallelization - instead of checking nibbles one by one,
; we load 64 bytes (128 nibbles) into a 512-bit register and compare them all at once!
;
; how it works:
; 1. load 64 bytes from read sequence into zmm0 (512-bit register)
; 2. load 64 bytes from ref sequence into zmm1 (512-bit register)
; 3. compare all 64 bytes simultaneously with vpcmpeqb
; 4. the result is a mask where each bit tells us if that byte matched
; 5. if all bits are set (all bytes match), we found 128 matching nibbles in one go!
;

; ok to summarize basically now we 11 loops
    ; main loop - process entire sequence (not avx-fied)
    ; shift loop - test all error hypotheses (not avx-fied)
    ; avx-512 - match on main diagonal (this is the avx-fied so this is the one parallelized)
    ; byte counting - count partial matches
    ; main scalar - fallback matching
    ; avx-512 upper (right diagonal) - below are js the subfunctions basta this is parallelized
        ; upper count (right diagonal) - count upper partials
        ; upper scalar - jic the upper loop goes wonky
    ; avx-512 upper (right diagonal) - below are js the subfunctions basta this is parallelized
        ; upper count (right diagonal) - count lower partials
        ; upper scalar - jic the lower loop goes wonky
; -------------------------------------------------------------------------------------

; returns: rax = number of consecutive matches
.check_main_diagonal_avx512:
    push    rbx
    push    rcx
    push    rdx
    push    rsi
    push    rdi
    push    r8
    push    r9
    
    xor     rax, rax              ; match_count = 0

    ; Create and apply mask for low 4 bits (0x0F0F0F0F...)
    mov     r8d, 0x0F0F0F0F
    vpbroadcastd zmm6, r8d
    
.avx512_main_loop:
    ; calculate current position
    mov     rbx, r15
    add     rbx, rax              ; current_pos = index + match_count
    
    ; check if we've reached the end
    cmp     rbx, r13
    jae     .avx512_main_done
    
    ; check if we're byte-aligned (nibble offset is even)
    ; avx-512 works best with aligned data
    test    bl, 1
    jnz     .avx512_main_scalar   ; if odd offset, use scalar fallback
    
    ; check if we have at least 128 nibbles (64 bytes) remaining
    ; this is the minimum we need for a full avx-512 operation
    mov     rcx, r13
    sub     rcx, rbx
    cmp     rcx, 128
    jb      .avx512_main_scalar   ; not enough data, use scalar
    
    ; calculate byte offset (divide nibble offset by 2)
    mov     rsi, rbx
    shr     rsi, 1
    
    ; *** load 64 bytes from each sequence into 512-bit zmm registers ***
    vmovdqu8 zmm0, [r11 + rsi]    ; read sequence (64 bytes at once!)
    vmovdqu8 zmm1, [r12 + rsi]    ; ref sequence (64 bytes at once!)
    
    ; Extract high nibbles 
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1
    vpsrld  zmm2, zmm2, 4         ; shift right to get high nibbles
    vpsrld  zmm3, zmm3, 4

    vpandd  zmm2, zmm2, zmm6      ; mask high nibbles to 4 bits
    vpandd  zmm3, zmm3, zmm6

    ; Extract low nibbles 
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd  zmm4, zmm4, zmm6      ; mask to get low nibbles
    vpandd  zmm5, zmm5, zmm6
    
    ; Compare nibbles
    vpcmpeqb k3, zmm2, zmm3       ; k3 = high nibble matches (even positions)
    vpcmpeqb k4, zmm4, zmm5       ; k4 = low nibble matches (odd positions)
    
    ; Invert to get mismatch masks (1 = mismatch, 0 = match)
    knotq   k3, k3                ; k3 = high nibble mismatches
    knotq   k4, k4                ; k4 = low nibble mismatches
    
    ; Check if all nibbles matched
    korq    k5, k3, k4            ; k5 = any mismatch in either nibble
    ktestq  k5, k5
    jz      .all_128_matched      ; if k5 is zero, all 128 nibbles matched!
    
    ;find mismatch
    ; Convert masks to 64-bit integers
    kmovq   rbx, k3               ; rbx = high nibble (even) mismatches
    kmovq   r10, k4               ; r10 = low nibble (odd) mismatches
    
    ; Find first high nibble mismatch (even positions: 0, 2, 4, 6...)
    tzcnt   r8, rbx               ; r8 = byte index of first high nibble mismatch
    jc      .no_high_mismatch     ; if carry, no mismatch found
    shl     r8, 1                 ; nibble index = byte * 2 (even position)
    jmp     .have_high_index
    
.no_high_mismatch:
    mov     r8, 128               ; no high nibble mismatch
    
.have_high_index:
    ; Find first low nibble mismatch (odd positions: 1, 3, 5, 7...)
    tzcnt   r9, r10               ; r9 = byte index of first low nibble mismatch
    jc      .no_low_mismatch      ; if carry, no mismatch found
    lea     r9, [r9*2 + 1]        ; nibble index = byte * 2 + 1 (odd position)
    jmp     .have_low_index
    
.no_low_mismatch:
    mov     r9, 129               ; no low nibble mismatch
    
.have_low_index:
    ; Take the minimum (first overall mismatch)
    cmp     r8, r9
    cmovbe  rdx, r8               ; rdx = min(r8, r9)
    cmova   rdx, r9
    
    ; Add matched nibbles to our counter
    add     rax, rdx
    jmp     .avx512_main_done
    
.all_128_matched:
    ; All 128 nibbles matched!
    add     rax, 128
    jmp     .avx512_main_loop     ; check the next 128 nibbles

.avx512_main_scalar:
    ; scalar fallback for when we cannot use avx-512
    ; this happens when:
    ; - at an odd nibble offset (not byte-aligned)
    ; - we have fewer than 128 nibbles remaining
    ; in these cases, we fall back to checking one nibble at a time
    mov     rbx, r15
    add     rbx, rax
    
    cmp     rbx, r13
    jae     .avx512_main_done
    
    ; extract the nibble from the read sequence
    mov     rcx, rbx
    shr     rcx, 1                 ; convert nibble offset to byte offset
    movzx   edi, byte [r11 + rcx]  ; load the byte
    test    bl, 1                  ; check if we want the high or low nibble
    jz      .avx512_read_even
    and     dil, 0x0F              ; odd offset = low nibble (bits 0-3)
    jmp     .avx512_read_done
.avx512_read_even:
    shr     dil, 4                 ; even offset = high nibble (bits 4-7)
.avx512_read_done:
    
    ; extract the nibble from the ref sequence
    mov     rcx, rbx
    shr     rcx, 1
    movzx   esi, byte [r12 + rcx]
    test    bl, 1
    jz      .avx512_ref_even
    and     sil, 0x0F
    jmp     .avx512_ref_done
.avx512_ref_even:
    shr     sil, 4
.avx512_ref_done:
    
    ; compare the two nibbles
    cmp     dil, sil
    jne     .avx512_main_done      ; mismatch found, stop
    
    inc     rax                    ; nibbles match, increment counter
    jmp     .avx512_main_loop      ; continue (might switch back to avx-512 if aligned)
    
.avx512_main_done:
    vzeroupper                     ; clean up avx state (important for performance)
    pop     r9
    pop     r8
    pop     rdi
    pop     rsi
    pop     rdx
    pop     rcx
    pop     rbx
    ret

; avx-512 optimized upper diagonal checker
; this checks for deletions in the reference sequence
; the parallelization works the same way as the main diagonal:
; - load 64 bytes from shifted positions
; - compare all 64 bytes at once with vpcmpeqb
; - count matches
;
; the difference is that we're comparing:
; - read[position - shift] with ref[position]
; this simulates a deletion by shifting the read backwards
;
; r9 = shift amount (how many positions to shift)
; returns: rax = number of consecutive matches
.check_upper_diagonal_avx512:
    push    rbx
    push    rcx
    push    rdx
    push    rsi
    push    rdi
    push    r8
    push    r9                   
    
    xor     rax, rax              ; match_count = 0
    mov     [rbp-32], r9          
    
.avx512_upper_loop:
    mov     rbx, r15
    add     rbx, rax              ; current ref position
    
    cmp     rbx, r13
    jae     .avx512_upper_done
    
    ; check if byte-aligned and have enough data for avx-512
    test    bl, 1
    jnz     .avx512_upper_scalar
    
    mov     rcx, r13
    sub     rcx, rbx
    cmp     rcx, 128
    jb      .avx512_upper_scalar
    
    ; calculate read position with shift (deletion)
    mov     rsi, rbx
    mov     r9, [rbp-32]          ; restore shift amount
    sub     rsi, r9               ; read_pos = ref_pos - shift
    test    rsi, rsi
    js      .avx512_upper_done    ; negative position, out of bounds
    
    ; make sure we have enough read data remaining
    mov     rdx, r13
    sub     rdx, rsi
    cmp     rdx, 128
    jb      .avx512_upper_scalar
    
    ; *** parallelized comparison with shift ***
    ; load 64 bytes from the shifted read position
    mov     r8, rsi
    shr     r8, 1
    vmovdqu8 zmm0, [r11 + r8]     ; read (shifted by deletion amount)
    
    ; load 64 bytes from the normal ref position
    mov     r8, rbx
    shr     r8, 1
    vmovdqu8 zmm1, [r12 + r8]     ; ref
    
    ; Extract high nibbles 
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1
    vpsrld  zmm2, zmm2, 4         ; shift right to get high nibbles
    vpsrld  zmm3, zmm3, 4

    vpandd  zmm2, zmm2, zmm6      ; mask high nibbles to 4 bits
    vpandd  zmm3, zmm3, zmm6

    ; Extract low nibbles 
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd  zmm4, zmm4, zmm6      ; mask to get low nibbles
    vpandd  zmm5, zmm5, zmm6
    
    ; Compare nibbles
    vpcmpeqb k3, zmm2, zmm3       ; k3 = high nibble matches (even positions)
    vpcmpeqb k4, zmm4, zmm5       ; k4 = low nibble matches (odd positions)
    
    ; Invert to get mismatch masks (1 = mismatch, 0 = match)
    knotq   k3, k3                ; k3 = high nibble mismatches
    knotq   k4, k4                ; k4 = low nibble mismatches
    
    ; Check if all nibbles matched
    korq    k5, k3, k4            ; k5 = any mismatch in either nibble
    ktestq  k5, k5
    jz      .all_128_matched_upper      ; if k5 is zero, all 128 nibbles matched!
    
    ; Find mismatch
    ; Convert masks to 64-bit integers
    kmovq   r8, k3                ; r8 = high nibble (even) mismatches
    kmovq   r10, k4               ; r10 = low nibble (odd) mismatches
    
    ; Find first high nibble mismatch (even positions: 0, 2, 4, 6...)
    tzcnt   rcx, r8               ; rcx = byte index 
    jc      .no_high_mismatch_upper
    shl     rcx, 1                ; nibble index = byte * 2 (even position)
    jmp     .have_high_index_upper
    
.no_high_mismatch_upper:
    mov     rcx, 128              ; no high nibble mismatch
    
.have_high_index_upper:
    ; Find first low nibble mismatch (odd positions: 1, 3, 5, 7...)
    tzcnt   rdx, r10              ; rdx = byte index 
    jc      .no_low_mismatch_upper
    lea     rdx, [rdx*2 + 1]      ; nibble index = byte * 2 + 1 (odd position)
    jmp     .have_low_index_upper
    
.no_low_mismatch_upper:
    mov     rdx, 129              ; no low nibble mismatch
    
.have_low_index_upper:
    ; Take the minimum (first overall mismatch)
    cmp     rcx, rdx              
    cmovbe  rsi, rcx              
    cmova   rsi, rdx
    
    ; Add matched nibbles to our counter
    add     rax, rsi              
    jmp     .avx512_upper_done
    
.all_128_matched_upper:
    ; All 128 nibbles matched!
    add     rax, 128
    jmp     .avx512_upper_loop     ; check the next 128 nibbles
    
.avx512_upper_scalar:
    ; scalar fallback for upper diagonal
    mov     rbx, r15
    add     rbx, rax
    
    cmp     rbx, r13
    jae     .avx512_upper_done
    
    ; calculate shifted read position
    mov     rsi, rbx
    mov     r9, [rbp-32]          ; restore shift amount
    sub     rsi, r9
    test    rsi, rsi
    js      .avx512_upper_done
    cmp     rsi, r13
    jae     .avx512_upper_done
    
    ; get nibbles and compare (same as main diagonal, but with shifted read)
    mov     rcx, rsi
    shr     rcx, 1
    movzx   edi, byte [r11 + rcx]
    test    sil, 1
    jz      .upper_read_even
    and     dil, 0x0F
    jmp     .upper_read_done
.upper_read_even:
    shr     dil, 4
.upper_read_done:
    
    mov     rcx, rbx
    shr     rcx, 1
    movzx   esi, byte [r12 + rcx]
    test    bl, 1
    jz      .upper_ref_even
    and     sil, 0x0F
    jmp     .upper_ref_done
.upper_ref_even:
    shr     sil, 4
.upper_ref_done:
    
    cmp     dil, sil
    jne     .avx512_upper_done
    
    inc     rax
    jmp     .avx512_upper_loop
    
.avx512_upper_done:
    vzeroupper
    pop     r9                    
    pop     r8
    pop     rdi
    pop     rsi
    pop     rdx
    pop     rcx
    pop     rbx
    ret

; avx-512 optimized lower diagonal checker
; this checks for insertions in the reference sequence
; the parallelization works the same way:
; - load 64 bytes from shifted positions
; - compare all 64 bytes at once
;
; the difference is that we're comparing:
; - read[position + shift] with ref[position]
; this simulates an insertion by shifting the read forwards
;
; r9 = shift amount
; returns: rax = number of consecutive matches
.check_lower_diagonal_avx512:
    push    rbx
    push    rcx
    push    rdx
    push    rsi
    push    rdi
    push    r8
    push    r9                    
    
    xor     rax, rax
    mov     [rbp-40], r9          
    
.avx512_lower_loop:
    mov     rbx, r15
    add     rbx, rax
    
    cmp     rbx, r13
    jae     .avx512_lower_done
    
    ; check if byte-aligned and enough data
    test    bl, 1
    jnz     .avx512_lower_scalar
    
    mov     rcx, r13
    sub     rcx, rbx
    cmp     rcx, 128
    jb      .avx512_lower_scalar
    
    ; calculate read position with shift (insertion)
    mov     rsi, rbx
    mov     r9, [rbp-40]          ; restore shift amount
    add     rsi, r9               ; read_pos = ref_pos + shift
    cmp     rsi, r13
    jae     .avx512_lower_done
    
    mov     rdx, r13
    sub     rdx, rsi
    cmp     rdx, 128
    jb      .avx512_lower_scalar
    
    ; *** parallelized comparison with forward shift ***
    ; load 64 bytes from the shifted read position
    mov     r8, rsi
    shr     r8, 1
    vmovdqu8 zmm0, [r11 + r8]     ; read (shifted forward by insertion amount)
    
    ; load 64 bytes from the normal ref position
    mov     r8, rbx
    shr     r8, 1
    vmovdqu8 zmm1, [r12 + r8]     ; ref
    
    ; *** NIBBLE-LEVEL EXTRACTION AND COMPARISON ***
    ; Extract high nibbles 
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1
    vpsrld  zmm2, zmm2, 4         ; shift right to get high nibbles
    vpsrld  zmm3, zmm3, 4

    vpandd  zmm2, zmm2, zmm6      ; mask high nibbles to 4 bits
    vpandd  zmm3, zmm3, zmm6

    ; Extract low nibbles 
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd  zmm4, zmm4, zmm6      ; mask to get low nibbles
    vpandd  zmm5, zmm5, zmm6
    
    ; Compare nibbles
    vpcmpeqb k3, zmm2, zmm3       ; k3 = high nibble matches (even positions)
    vpcmpeqb k4, zmm4, zmm5       ; k4 = low nibble matches (odd positions)
    
    ; Invert to get mismatch masks (1 = mismatch, 0 = match)
    knotq   k3, k3                ; k3 = high nibble mismatches
    knotq   k4, k4                ; k4 = low nibble mismatches
    
    ; Check if all nibbles matched
    korq    k5, k3, k4            ; k5 = any mismatch in either nibble
    ktestq  k5, k5
    jz      .all_128_matched_lower      ; if k5 is zero, all 128 nibbles matched!
    
    ; Find mismatch
    ; Convert masks to 64-bit integers
    kmovq   r8, k3                ; r8 = high nibble (even) mismatches
    kmovq   r10, k4               ; r10 = low nibble (odd) mismatches
    
    ; Find first high nibble mismatch (even positions: 0, 2, 4, 6...)
    tzcnt   rcx, r8               ; rcx = byte index
    jc      .no_high_mismatch_lower
    shl     rcx, 1                ; nibble index = byte * 2 (even position)
    jmp     .have_high_index_lower
    
.no_high_mismatch_lower:
    mov     rcx, 128              ; no high nibble mismatch
    
.have_high_index_lower:
    ; Find first low nibble mismatch (odd positions: 1, 3, 5, 7...)
    tzcnt   rdx, r10              ; rdx = byte index
    jc      .no_low_mismatch_lower
    lea     rdx, [rdx*2 + 1]      ; nibble index = byte * 2 + 1 (odd position)
    jmp     .have_low_index_lower
    
.no_low_mismatch_lower:
    mov     rdx, 129              ; no low nibble mismatch
    
.have_low_index_lower:
    ; Take the minimum (first overall mismatch)
    cmp     rcx, rdx
    cmovbe  rsi, rcx              ; rsi = min(rcx, rdx)
    cmova   rsi, rdx
    
    ; Add matched nibbles to our counter
    add     rax, rsi
    jmp     .avx512_lower_done
    
.all_128_matched_lower:
    ; All 128 nibbles matched!
    add     rax, 128
    jmp     .avx512_lower_loop
    
.avx512_lower_scalar:
    ; scalar fallback for lower diagonal
    mov     rbx, r15
    add     rbx, rax
    
    cmp     rbx, r13
    jae     .avx512_lower_done
    
    ; calculate shifted read position
    mov     rsi, rbx
    mov     r9, [rbp-40]        
    add     rsi, r9
    cmp     rsi, r13
    jae     .avx512_lower_done
    
    ; get nibbles and compare
    mov     rcx, rsi
    shr     rcx, 1
    movzx   edi, byte [r11 + rcx]
    test    sil, 1
    jz      .lower_read_even
    and     dil, 0x0F
    jmp     .lower_read_done
.lower_read_even:
    shr     dil, 4
.lower_read_done:
    
    mov     rcx, rbx
    shr     rcx, 1
    movzx   esi, byte [r12 + rcx]
    test    bl, 1
    jz      .lower_ref_even
    and     sil, 0x0F
    jmp     .lower_ref_done
.lower_ref_even:
    shr     sil, 4
.lower_ref_done:
    
    cmp     dil, sil
    jne     .avx512_lower_done
    
    inc     rax
    jmp     .avx512_lower_loop
    
.avx512_lower_done:
    vzeroupper
    pop     r9                    
    pop     r8
    pop     rdi
    pop     rsi
    pop     rdx
    pop     rcx
    pop     rbx
    ret

Overwriting checkpointavxfull.asm


In [3]:
%%writefile checkpointavxfull.asm

default rel
bits 64

; ==============================================================================
; TIMING MACROS
; ==============================================================================
; Structure offsets based on C definition (Timer struct = 24 bytes)
%define OFF_MAIN_DIAG  24
%define OFF_RIGHT_DIAG 48
%define OFF_LEFT_DIAG  72
%define OFF_TOTAL      96

; Macro to start a timer
%macro START_TIMER 1
    push    rax
    push    rdx
    rdtsc                   ; Time in EDX:EAX
    shl     rdx, 32
    or      rax, rdx        ; Combine to 64-bit in RAX
    mov     %1, rax         ; Save start time to stack
    pop     rdx
    pop     rax
%endmacro

; Macro to stop a timer and accumulate results
%macro STOP_TIMER 2
    push    rax
    push    rdx
    push    rcx             ; Save RCX (often used as temp)
    
    rdtsc                   ; Get end time
    shl     rdx, 32
    or      rax, rdx
    
    sub     rax, %1         ; RAX = End - Start (Delta)
    
    ; Load address of timing_data
    lea     rcx, [timing_data + %2]
    
    ; Add delta to total_cycles (offset 8 in Timer struct)
    add     [rcx + 8], rax
    
    ; Increment call_count (offset 16 in Timer struct)
    inc     qword [rcx + 16]
    
    pop     rcx
    pop     rdx
    pop     rax
%endmacro
    
section .data
global SneakySnake
global current_position
global current_edits
global mismatch_count
global safety_counter

; Import the C struct
extern timing_data

current_position dq 0
current_edits dq 0
mismatch_count dq 0
safety_counter dq 0

section .text
global SneakySnake

SneakySnake:
    push rbp
    mov rbp, rsp
    push rbx
    push r12
    push r13
    push r14
    push r15
    sub rsp, 64

    START_TIMER [rbp-56] ;start timer

    ; parameters:
    ; rdi = readlength (in nibbles)
    ; rsi = refseq
    ; rdx = readseq
    ; rcx = editthreshold
    ; r8  = iterationno

    ; Initialize counters
    xor     rax, rax
    mov     [current_position], rax
    mov     [current_edits], rax
    mov     [mismatch_count], rax
    mov     [safety_counter], rax

    mov     r13, rdi              ; readlength
    mov     r12, rsi              ; refseq pointer
    mov     r11, rdx              ; readseq pointer
    mov     r10, rcx              ; editthreshold
    mov     [rbp-8], r8           ; iterationno

    xor     r15, r15              ; index = 0
    xor     r14, r14              ; Edits = 0
   ; mov     qword [rbp-32], 1     ; roundsNo = 1
    
    ; to extract nibbles later
    mov     r8d, 0x0F0F0F0F
    vpbroadcastd zmm6, r8d        ; zmm6 = nibble mask 

.while_loop:
    cmp r15, r13                  ; cmp with readlen
    jae .accept

     ; safety check to prevent infinite loops
    inc     qword [safety_counter]
    mov     rax, [safety_counter]
    cmp     rax, [rbp-8]
    jg      .reject

    ; cmp with edit threshold
    cmp r14, r10
    jg .reject

    START_TIMER [rbp-32]

    xor rbx, rbx                  ; match_count = 0

;------------------- MAIN DIAGONAL ---------------------
.main_diag:
    mov rax, r15
    add rax, rbx                  ; position = index + match_count

    cmp rax, r13                  ; cmp with readlen
    jae .main_done

    test al, 1
    jnz .main_scalar              ; check if odd

    mov rcx, r13
    sub rcx, rax
    cmp rcx, 128
    jb .main_scalar               ; check if <128 nibbles

    mov rsi, rax
    shr rsi, 1                    ; calc byte (/2)

    ; prefetch
    prefetcht0 [r11 + rsi + 64]
    prefetcht0 [r12 + rsi + 64]

    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + rsi]    ; load read and ref

    ; get high nibbles (even)
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1

    vpsrld zmm2, zmm2, 4
    vpsrld zmm3, zmm3, 4

    vpandd zmm2, zmm2, zmm6
    vpandd zmm3, zmm3, zmm6

    ; get low nibbles (odd)
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd zmm4, zmm4, zmm6
    vpandd zmm5, zmm5, zmm6

    ; cmp read and ref
    vpcmpeqb k3, zmm2, zmm3       
    vpcmpeqb k4, zmm4, zmm5       

    ; invert (0-match/1-mismatch)
    knotq k3, k3
    knotq k4, k4

    ; check if all matched
    korq k5, k3, k4
    ktestq k5, k5
    jz .all_match

    kmovq r8, k3
    kmovq r9, k4

    ; find high mismatch
    tzcnt r8, r8
    shl r8, 1

    ; find low mismatch
    tzcnt r9, r9
    lea r9, [r9*2+1]

    ; get minimum
    cmp r8, r9
    cmovbe rdx, r8
    cmova rdx, r9

    add rbx, rdx                  ; number of matches in main
    jmp .main_done

.all_match:
    add rbx, 128
    jmp .main_diag

.main_scalar:                     
    mov rax, r15
    add rax, rbx

    cmp rax, r13
    jae .main_done

    mov rcx, rax
    shr rcx, 1
    movzx edi, byte[r11+rcx]
    movzx esi, byte[r12+rcx]

    test al, 1
    jz .main_even
    and dil, 0x0F
    and sil, 0x0F
    jmp .main_cmp

.main_even:
    shr dil, 4
    shr sil, 4

.main_cmp:
    cmp dil, sil
    jne .main_done

    inc rbx
    jmp .main_diag

.main_done:

    STOP_TIMER [rbp-32], OFF_MAIN_DIAG
    
    mov rcx, r13
    sub rcx, r15
    cmp rbx, rcx
    jae .done

    mov [rbp-16], rbx             ; store global_count
    mov qword [rbp-24], 0         ; best shift

    mov r8, 1                     ; r8 = current edit distance/shift

.shift_loop:                      
    cmp r8, r10
    ja .shift_done

    START_TIMER [rbp-40]
    xor rax, rax                  ; count = 0

; ----------------------- UPPER DIAG ------------------------------

.upper_diag:
    mov r9, r15
    add r9, rax                   ; curr index

    cmp r9, r13                   ; cmp with readlen
    jae .upper_done

    test r9b, 1                   ; check if byte aligned
    jnz .upper_scalar
    
    mov rcx, r13
    sub rcx, r9
    cmp rcx, 128
    jb .upper_scalar              ; check if <128

    mov rsi, r9
    sub rsi, r8                   ; read = ref - shift
    test rsi, rsi
    js .upper_done

    ; check read position alignment
    test    sil, 1
    jnz     .upper_scalar

    ; check remaining read data
    mov rdx, r13
    sub rdx, rsi
    cmp rdx, 128
    jb .upper_scalar

    push r9
    mov r9, rsi
    shr r9, 1
    vmovdqu8 zmm0, [r11 + r9]
    pop r9

    push rsi
    mov rsi, r9
    shr rsi, 1
    vmovdqu8 zmm1, [r12 + rsi]
    pop rsi

    ; get high nibbles (even)
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1

    vpsrld zmm2, zmm2, 4
    vpsrld zmm3, zmm3, 4

    vpandd zmm2, zmm2, zmm6
    vpandd zmm3, zmm3, zmm6

    ; get low nibbles (odd)
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd zmm4, zmm4, zmm6
    vpandd zmm5, zmm5, zmm6

    ; cmp read and ref
    vpcmpeqb k3, zmm2, zmm3       
    vpcmpeqb k4, zmm4, zmm5       

    ; invert (0-match/1-mismatch)
    knotq k3, k3
    knotq k4, k4

    ; check if all matched
    korq k5, k3, k4
    ktestq k5, k5
    jz .all_upper_match

    kmovq rcx, k3
    kmovq rdx, k4

    ; find high mismatch
    tzcnt rcx, rcx
    shl rcx, 1

    ; find low mismatch
    tzcnt rdx, rdx
    lea rdx, [rdx*2+1]

    ; get minimum
    cmp rcx, rdx
    cmovbe rdi, rcx
    cmova rdi, rdx

    add rax, rdi
    jmp .upper_done

.all_upper_match:
    add rax, 128
    jmp .upper_diag

.upper_scalar:
    mov r9, r15
    add r9, rax

    cmp r9, r13
    jae .upper_done

    mov rsi, r9
    sub rsi, r8
    test rsi, rsi
    js .upper_done
    cmp rsi, r13
    jae .upper_done

    ; cmp nibbles
    mov rcx, rsi
    shr rcx, 1
    movzx edi, byte [r11 + rcx]
    test sil, 1
    jz .upper_read_even
    and dil, 0x0F
    jmp .upper_read_done

.upper_read_even:
    shr dil, 4

.upper_read_done:
    mov rcx, r9
    shr rcx, 1
    movzx esi, byte [r12 + rcx]
    test r9b, 1
    jz .upper_ref_even
    and sil, 0x0F
    jmp .upper_ref_done

.upper_ref_even:
    shr sil, 4

.upper_ref_done:
    cmp dil, sil
    jne .upper_done
    inc rax
    jmp .upper_diag

.upper_done:

    STOP_TIMER [rbp-40], OFF_RIGHT_DIAG

    mov rcx, r13
    sub rcx, r15
    cmp rax, rcx
    jae .done

    ; update if current is better
    mov rdx, [rbp-16]
    cmp rax, rdx
    jbe .lower_loop

    mov [rbp-16], rax
    mov [rbp-24], r8

.lower_loop:
    START_TIMER [rbp-48]

    xor rax, rax                  ; count = 0
; ------------------------- LOWER DIAG --------------------------
.lower_diag:
    
    mov r9, r15
    add r9, rax                   

    cmp r9, r13
    jae .lower_done

    test r9b, 1
    jnz .lower_scalar

    mov rcx, r13
    sub rcx, r9
    cmp rcx, 128
    jb .lower_scalar

    ; shift read
    mov rsi, r9
    add rsi, r8
    cmp rsi, r13
    jae .lower_done

    ; check read position alignment
    test    sil, 1
    jnz     .lower_scalar

    ; check remaining read data
    mov rdx, r13
    sub rdx, rsi
    cmp rdx, 128
    jb .lower_scalar

    push r9
    mov r9, rsi
    shr r9, 1
    vmovdqu8 zmm0, [r11 + r9]
    pop r9

    push rsi
    mov rsi, r9
    shr rsi, 1
    vmovdqu8 zmm1, [r12 + rsi]
    pop rsi

    ; get high nibbles (even)
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1

    vpsrld zmm2, zmm2, 4
    vpsrld zmm3, zmm3, 4

    vpandd zmm2, zmm2, zmm6
    vpandd zmm3, zmm3, zmm6

    ; get low nibbles (odd)
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd zmm4, zmm4, zmm6
    vpandd zmm5, zmm5, zmm6

    ; cmp read and ref
    vpcmpeqb k3, zmm2, zmm3       
    vpcmpeqb k4, zmm4, zmm5       

    ; invert (0-match/1-mismatch)
    knotq k3, k3
    knotq k4, k4

    ; check if all matched
    korq k5, k3, k4
    ktestq k5, k5
    jz .all_lower_match

    kmovq rcx, k3
    kmovq rdx, k4

    ; find high mismatch
    tzcnt rcx, rcx
    shl rcx, 1

    ; find low mismatch
    tzcnt rdx, rdx
    lea rdx, [rdx*2+1]

    ; get minimum
    cmp rcx, rdx
    cmovbe rdi, rcx
    cmova rdi, rdx

    add rax, rdi
    jmp .lower_done

.all_lower_match:
    add rax, 128
    jmp .lower_diag

.lower_scalar:
    mov r9, r15
    add r9, rax

    cmp r9, r13
    jae .lower_done

    mov rsi, r9
    add rsi, r8                   
    cmp rsi, r13
    jae .lower_done

    ; cmp nibbles
    mov rcx, rsi
    shr rcx, 1
    movzx edi, byte [r11 + rcx]
    test sil, 1
    jz .lower_read_even           
    and dil, 0x0F
    jmp .lower_read_done

.lower_read_even:
    shr dil, 4

.lower_read_done:
    mov rcx, r9
    shr rcx, 1
    movzx esi, byte [r12 + rcx]
    test r9b, 1
    jz .lower_ref_even
    and sil, 0x0F
    jmp .lower_ref_done

.lower_ref_even:
    shr sil, 4

.lower_ref_done:
    cmp dil, sil
    jne .lower_done
    inc rax
    jmp .lower_diag

.lower_done:

    STOP_TIMER [rbp-48], OFF_LEFT_DIAG

    mov rcx, r13
    sub rcx, r15
    cmp rax, rcx
    jae .done

    ; update counter if current is better
    mov rdx, [rbp-16]
    cmp rax, rdx
    jbe .next_shift               

    mov [rbp-16], rax
    mov r9, r8
    neg r9
    mov [rbp-24], r9

.next_shift:                      
    inc r8
    jmp .shift_loop

.shift_done:
    ; add best match count
    mov rbx, [rbp-16]
    add r15, rbx

    cmp r15, r13
    jae .while_loop

    inc r14                       ; edits++
    inc r15                       ; index++

    inc qword [rbp-32]            
    jmp .while_loop               

.done:
    mov r15, r13
    jmp .while_loop

.accept:
    mov [current_position], r13
    mov [current_edits], r14
    cmp r14, r10
    jg .reject
    mov rax, 1
    jmp .end

.reject:
    xor rax, rax

.end:
    STOP_TIMER [rbp-56], OFF_TOTAL

    vzeroupper
    add rsp, 64
    pop r15
    pop r14
    pop r13
    pop r12
    pop rbx
    leave
    ret

Overwriting checkpointavxfull.asm


In [1]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 10 100 30000000 0

Loading sequences (Fixed Len=100, Limit=30000000)...
Loaded 30000000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 30000000
Fixed Length             : 100
Threshold                : 10

C Implementation:
  Accepted               : 22631061
  Time                   : 192.2863 seconds

ASM Implementation:
  Accepted               : 22631061
  Time                   : 127.5524 seconds
  Speedup vs C           : 1.51x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake            81.140205     30000000           10819
  Main Diagonal              21.409698    229525552             373
  Right (Upper) Diag         16.105244   2109810757              31
  Left (Lower) Diag          16.409004   2106985828              31
  Other/Over

In [9]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/ERR240727_1_E40_30million.txt" 1 100 30000000 0

Loading sequences (Fixed Len=100, Limit=30000000)...
^Caded 4111000 sequences...


In [40]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 1 100 30000000 0

Loading sequences (limit=30000000)...
Loaded 30000000 sequences...
Loaded 30000000 sequences. Ground truth matches (edit_dist <= 1): 1345842

=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

               BENCHMARK SUMMARY                  
Total Pairs              : 30000000
Ground Truth Matches     : 1345842
Threshold                : 1

C Implementation:
  Accepted               : 1358315
  Disagreements w/ truth : 12473
  Time                   : 107.2599 seconds

ASM Implementation:
  Accepted               : 1358315
  Disagreements w/ truth : 12473
  Time                   : 25.0411 seconds
  Speedup vs C           : 4.28x



KeyboardInterrupt



In [41]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 5 100 30000000 0

Loading sequences (limit=30000000)...
Loaded 30000000 sequences...
Loaded 30000000 sequences. Ground truth matches (edit_dist <= 5): 9821308

=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

               BENCHMARK SUMMARY                  
Total Pairs              : 30000000
Ground Truth Matches     : 9821308
Threshold                : 5

C Implementation:
  Accepted               : 10616890
  Disagreements w/ truth : 795582
  Time                   : 140.1895 seconds

ASM Implementation:
  Accepted               : 10616890
  Disagreements w/ truth : 795582
  Time                   : 41.9887 seconds
  Speedup vs C           : 3.34x


In [42]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 10 100 30000000 0

Loading sequences (limit=30000000)...
Loaded 30000000 sequences...
Loaded 30000000 sequences. Ground truth matches (edit_dist <= 10): 18610897

=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

               BENCHMARK SUMMARY                  
Total Pairs              : 30000000
Ground Truth Matches     : 18610897
Threshold                : 10

C Implementation:
  Accepted               : 22631061
  Disagreements w/ truth : 4020164
  Time                   : 199.0949 seconds

ASM Implementation:
  Accepted               : 22631061
  Disagreements w/ truth : 4020164
  Time                   : 77.1828 seconds
  Speedup vs C           : 2.58x


In [ ]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 0 250 30000000 0

Loading sequences (limit=30000000)...
Loaded 195000 sequences...

In [ ]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 1 250 30000000 0

Loading sequences (limit=30000000)...
Loaded 135000 sequences...

In [ ]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 5 250 30000000 0

Loading sequences (limit=30000000)...
Loaded 135000 sequences...

In [ ]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/SRR826471_1_E8_30million.txt" 10 250 30000000 0

Loading sequences (limit=30000000)...
Loaded 135000 sequences...

In [91]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/LongSequences_10K_PBSIM_1KPairs.txt" 10 10000 1000 0

Loading sequences (Fixed Len=10000, Limit=1000)...
Loaded 1000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 1000
Fixed Length             : 10000
Threshold                : 10

C Implementation:
  Accepted               : 0
  Time                   : 0.0077 seconds

ASM Implementation:
  Accepted               : 0
  Time                   : 0.0061 seconds
  Speedup vs C           : 1.27x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake             0.003642         1000           14568
  Main Diagonal               0.000105        11000              38
  Right (Upper) Diag          0.000933       110000              34
  Left (Lower) Diag           0.000910       110000              33
  Other/Overhead              0.001694

In [92]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/LongSequences_100K_PBSIM_1KPairs.txt" 10 100000 1000 0

Loading sequences (Fixed Len=100000, Limit=1000)...
Loaded 1000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 1000
Fixed Length             : 100000
Threshold                : 10

C Implementation:
  Accepted               : 0
  Time                   : 0.0075 seconds

ASM Implementation:
  Accepted               : 0
  Time                   : 0.0057 seconds
  Speedup vs C           : 1.30x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake             0.003431         1000           13725
  Main Diagonal               0.000088        11000              32
  Right (Upper) Diag          0.000892       110000              32
  Left (Lower) Diag           0.000878       110000              32
  Other/Overhead              0.0015

In [13]:
!nasm -f elf64 checkpointavxfull.asm -o checkpointavxfull.o 
!gcc -c checkpointfull.c -o checkpointfull.o -mavx512f -mavx512bw 
!gcc checkpointavxfull.o checkpointfull.o -o checkpoint1 -mavx512f -mavx512bw 
!./checkpoint1 "../THES3/DNAPAIRS/Ecoli_Reads_10bp.txt" 10 10 30000 0

Loading sequences (Fixed Len=10, Limit=30000)...
Loaded 30000 sequences...
=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 30000
Fixed Length             : 10
Threshold                : 10

C Implementation:
  Accepted               : 30000
  Time                   : 0.0305 seconds

ASM Implementation:
  Accepted               : 30000
  Time                   : 0.0529 seconds
  Speedup vs C           : 0.58x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                    Total (s)        Calls      Avg Cycles
----------------------------------------------------------------------------
Total SneakySnake             0.032498        30000            4333
  Main Diagonal               0.001273       125727              41
  Right (Upper) Diag          0.007537      1125764              27
  Left (Lower) Diag           0.008398      1112023              30
  Other/Overhead              0.0